# Hosting Capacity Validation, PG&E

In order to validate that our hosting capacity calculations are correct, we will compare them to the hosting capacity maps created by Brockway et al.

This notebook recreates maps for the PG&E sample (below).

**Figure d:** PV, **Figure e:** PV w OpFlex

In [1]:
# bounding box (south west north east)
# map = map.bbox([37.5, -121.9, 38.4, -121.1])

In [2]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import box
from shapely.geometry import MultiLineString
import matplotlib.pyplot as plt

In [3]:
os.environ['PROJ_LIB'] = '/opt/anaconda3/share/proj'

In [4]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

## Read in Data

### PG&E Utility

In [5]:
# read in feederline data
pge_circuits = gpd.read_file("../../../../../capstone/electrigrid/data/utilities/pge_shapefiles/LineDetail/LineDetail.shp").to_crs("EPSG:4326")

# load in shapefile for extent of PG&E
utility_ter = gpd.read_file("../../../../../capstone/electrigrid/data/utilities/IOU_shapefiles.geojson")

#### Convert `GenCapacit` to MW 

It is currently in KW. We convert in order to standardize to the units of the other two utilities (SDG&E and SCE).

In [6]:
pge_circuits['GenCapacit'] = pge_circuits['GenCapacit'] / 1000
pge_circuits['GenCapac_1'] = pge_circuits['GenCapac_1'] / 1000

#### Change column names to match other utilites

In [7]:
pge_circuits = pge_circuits.rename(columns = {'FeederId':'circuit_id',
                                              'CSV_LineSe':'segment_id',
                                             'GenCapacit':'gencap',
                                             'GenCapac_1':'gencap_opflex',
                                             'GenericPVC':'gencap_pv',
                                             'GenericCap':'gencap_pv_opflex'})

In [8]:
pge_circuits.head(2)

,circuit_id,FeederName,globalid,segment_id,LoadCapaci,voltage_kv,phase_cnt,limiting_m,limiting_c,ICA_Analys,lica_analy,Division,gencap,gencap_pv,gencap_opflex,gencap_pv_opflex,limiting_1,limiting_2,limiting_3,limiting_4,limiting_5,limiting_6,limiting_7,limiting_8,ScreenL,Publish,Last_Updat,SHAPE_Leng,geometry
0,062541102,MERIDIAN 1102,{3F991049-BA44-489F-A403-DA79E95B5F6A},3862041,0,12.0,3,None,None,May 2024,May 2024,Sacramento,0.00,0,0.00,0,None,None,None,None,None,None,None,None,Unlikely to pass,1,2025-06-02,145.642120,"LINESTRING (-121.95921 39.12370, -121.95951 39..."
1,043302102,MONROE 2102,{65E86C65-2474-4DE9-831A-73C5C6C88469},5458148,2380,21.0,3,None,None,Feb 2025,Feb 2025,Sonoma,0.06,70,2.44,2440,None,None,None,None,None,None,None,None,Likely to pass,1,2025-06-02,9.102052,"LINESTRING (-122.73809 38.48070, -122.73809 38..."


### Census

In [9]:
# read in the census tract data
census_tracts = gpd.read_file("../../../../../capstone/electrigrid/data/census/tl_2025_06_tract/tl_2025_06_tract.shp").to_crs("EPSG:4326")

### Building

In [10]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
fp = "../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet"
building = gpd.read_parquet(fp).to_crs(epsg=4326)

### Zillow

In [ ]:
# import Zillow data (make take 10-20 minutes)
fp = "../../../../../capstone/electrigrid/data/buildings/final_zillow.gpkg"
zillow = gpd.read_file(fp).to_crs(epsg=4326)

We are currently operating with data just for Alameda County, which is under PG&E's jurisdiction. Therefore, instead of clipping to PG&E territory we will clip to the county, rendering the PG&E extent shapefile unnecessary (but only for now!).

#### Clip PG&E utility data to match the Brockway plot

In [ ]:
# create bounding box from what brockway map covered
brockway_box = box(-121.9, 37.5, -121.1, 38.4)

# clip circuits to that bounding box
pge_clipped = gpd.clip(pge_circuits, brockway_box)

# clip buildings
buildings_clipped = gpd.clip(building, brockway_box)

# clip census
census_clipped = gpd.clip(census_tracts, brockway_box)

## Preview data

Multi- and single-family homes in blue, feederlines in green, and census tract delineations in gray.

In [ ]:
# plot all of the data sources to ensure everything looks accurate 
fig, ax = plt.subplots(figsize=(10, 10))

ax.axis('off')

pge_clipped.plot(ax=ax, 
                    color='blue')

buildings_clipped.plot(ax=ax, 
                      color='green')

census_clipped.boundary.plot(ax=ax, 
                    color='gray')

## Keep only residential buildings

In [ ]:
# select only building footprints where a Zillow point exists
buildings_clipped = gpd.sjoin(
    buildings_clipped,
    zillow,
    how = "inner",
    predicate = "intersects")

In [ ]:
buildings_clipped.to_file("data/buildings_brock.geojson", driver='GeoJSON')

## Link Buildings to Feederlines

### Adding length column

Before joining the data frames and replacing the feederline geometries with those of the building, we will calculate the length of the lines and bring them over as a column in the join.

In [ ]:
# change the crs to a projected CRS
pge_clipped = pge_clipped.to_crs("EPSG:3310")
buildings_clipped = buildings_clipped.to_crs("EPSG:3310")

## create length column in metres
pge_clipped['length_m'] = pge_clipped.length

In [ ]:
pge_clipped.head(2)

#### All buildings

In [ ]:
# index the data
pge_clipped.sindex
buildings_clipped.sindex

# spatial join
pge_buildings_linked = gpd.sjoin_nearest(buildings_clipped, 
                                        pge_clipped, 
                                        how="left", 
                                        lsuffix='_left', 
                                        rsuffix='_right',
                                        distance_col='dist_to_line_m')

### Filter for homes that are less than 1000m away from the nearest feederline
If they are more than 1 km away, we assume they get their power from a different source.

In [ ]:

pge_buildings_linked = pge_buildings_linked[pge_buildings_linked['dist_to_line_m'] >= 1000]

## Hosting Capacity Calculation

### Step 2: Add census tract ID to each home

In [ ]:
## single family join

# change CRS so it matches
pge_buildings_linked = pge_buildings_linked.to_crs("EPSG:4326")

assert pge_buildings_linked.crs == census_clipped.crs

# drop columns that makes joins impossible
pge_buildings_linked = pge_buildings_linked.drop(['index_right'], axis = 1)

# join
pge_buildings_linked = pge_buildings_linked.sjoin(
    census_clipped,
    how = "left",
    predicate = "intersects")

In [ ]:
pge_buildings_linked.head(2)

### Step 3: Calculate the number of homes in each census tract

In [ ]:
# calculate the number of observations in each
by_tract = pge_buildings_linked.groupby('GEOID_right').sum('unit')

# save just the unit column and rename to be more intuitive
units_by_tract = by_tract['unit']
units_by_tract = units_by_tract.rename('units_by_tract')

# join units by tract to entire df
pge_buildings_linked = pge_buildings_linked.merge(
    units_by_tract,
    on = "GEOID_right")

### Step 4: Calculate the maximum ICA generation capacity for each circuit

In [ ]:
# find maximum generation across feederline
max_gen = pge_buildings_linked.groupby('circuit_id').max('GenCapacit')

# save just the necessary column and rename
max_gen_feeder = max_gen['gencap_pv'].rename('max_gen')

# join max generation by feeder to entire df
pge_buildings_linked = pge_buildings_linked.merge(
    max_gen_feeder,
    on = 'circuit_id')

### Step 5: Calculate the percentage of the length of each segment relative to the entire feederline/circuit

In [ ]:
# find the total length by Feeder Id
total_length = pge_buildings_linked.groupby('circuit_id').sum('length_m')

# rename
total_length = total_length['length_m'].rename('total_feeder_length')

# join
pge_buildings_linked = pge_buildings_linked.merge(
    total_length,
    on = 'circuit_id')

In [ ]:
# calculate percentage
pge_buildings_linked['perc_length'] = (pge_buildings_linked['length_m'] / pge_buildings_linked['total_feeder_length']) * 100

**Note:** Why are there multiple rows with the same `CSV_LineSe`?

### Step 6: Calculate the number of homes connected to each segment

In [ ]:
# find the number of homes
home_count_seg = pge_buildings_linked.groupby('segment_id').sum('unit')

# save just the unit columns and rename to be more intuitive
home_count_seg = home_count_seg['unit'].rename('home_count_seg')

# join with original df based on line segment 
pge_buildings_linked = apge_buildings_linked.merge(
    home_count_seg,
    on = "segment_id")

### Step 7: Calculate the number of homes connected to each circuit

In [ ]:
# find the number of homes
home_count_seg = pge_buildings_linked.groupby('circuit_id').sum('unit')

# save just the unit columns and rename to be more intuitive
home_count_seg = home_count_seg['unit'].rename('home_count_circuit')

# join with original df based on line segment 
pge_buildings_linked = pge_buildings_linked.merge(
    home_count_seg,
    on = "circuit_id")

### Step 8: Calculate the number of homes connected to segment relative to the number of homes connected to entire feederline/circuit

In [ ]:
# calculate perc of homes and save as column
pge_buildings_linked['perc_homes'] = (pge_buildings_linked['home_count_seg'] / pge_buildings_linked['home_count_circuit']) * 100

### Step 9: Calculate weighted generation capacity for each segment
Multiply the maximum generation capacity of each circuit by the the percentage of households each segment represents relative to the whole (`perc_homes`). This will undercount the total maximum hosting capacity of each circuit.

### Step 10: Calculate the number of homes located within each census tract for each circuit

**Variable:** # of households within circuit polygon

Circuit polygon definition: corresponds to the overlap between census tract and the estimated service area of circuit

In [ ]:
circ_poly = pge_buildings_linked.groupby(['GEOID_right', 'circuit_id']).sum('unit')

circ_poly_units = circ_poly['unit'].rename('tothh_Cpoly')

pge_buildings_linked = pge_buildings_linked.merge(
    circ_poly_units,
    on = ['GEOID_right', 'circuit_id'])

### Step 10b: Calculate the max generation within each census tract for each circuit

**Variable:** maximum generation within census tract, by feederline

In [ ]:
circ_poly_gen = pge_buildings_linked.groupby(['GEOID_right', 'circuit_id']).max('gencap_pv')

circ_poly_gen_max = circ_poly['gencap_pv'].rename('DER_max_Cpoly')

pge_buildings_linked = pge_buildings_linked.merge(
    circ_poly_gen_max,
    on = ['GEOID_right', 'circuit_id'])

### Step 11: Weight by household (complete Equation 8)

**Equation 8 from Brockway et al.**: 

_hhWt = (max capacity across circuit polygon) * (# of households within circuit polygon / # of households served by circuit)

**or**

`_hhWt` = `DER_max_Cpoly` * ( `tothh_Cpoly` / `home_count_circuit` )

In [ ]:
# calculate weight by household
pge_buildings_linked['_hhWt'] = pge_buildings_linked['DER_max_Cpoly'] * (pge_buildings_linked['tothh_Cpoly'] / pge_buildings_linked['home_count_circuit'])

**Note:** Values are different for observations with same `CSV_LineSe` but different GEOIDs.

### Step 12: Normalize household weight (complete Equation 9)

**Equation 9 from Brockway et al.:**

_hhWt_n = (household-weight max capacity) * (maximum capacity anywhere on circuit / household-weighted max capacity summed across circuit)

**or**

`_hhWt_n` = `_hhWt` * (`max_gen` / summ j [`_hhWt`] )

In [ ]:
# calculate summation of household-weighted max capacity across circuit
summ_hhWt = pge_buildings_linked.groupby('circuit_id').sum('_hhWt')

# save the summed column and rename
summ_hhWt = summ_hhWt['_hhWt'].rename('summ_hhWt')

# add to data frame through join
pge_buildings_linked = pge_buildings_linked.merge(
    summ_hhWt,
    on = ['circuit_id'])

In [ ]:
# equation 9
pge_buildings_linked['_hhWt_n'] = pge_buildings_linked['_hhWt'] * (pge_buildings_linked['max_gen'] / pge_buildings_linked['summ_hhWt'])

### Step 13: Adjust for observations where max capacity is exceeded (complete Equation 10)

"For a small minority of circuit polygons (0.05–1.60%, depending on DER type),
the normalized hosting capacity value exceeds the maximum allowed hosting
capacity in that circuit polygon. In these cases, we adjusted the value back to its
allowed maximum:"

**Equation 10 from Brockway et al.:

_hhWt_n = 

    if _hhWt_n > (max capacity across circuit polygon), then replace with max capacity across circuit polygon,
    else: 
            _hhWt_n
    
**or**

`_hhWt_n` = if `_hhWt_n` > `DER_max_Cpoly`, then `DER_max_Cpoly`,
            else `_hhWt_n`

In [ ]:
pge_buildings_linked['_hhWt_n'] = np.where(
    
    # condition -- weighted generation is greater than max generation across circuit/feederline
   pge_buildings_linked['_hhWt_n'] > pge_buildings_linked['DER_max_Cpoly'],
    
    # replace with max generation of feederline if condition is true
    pge_buildings_linked['DER_max_Cpoly'], 
    
    # otherwise, keep original calculated value
   pge_buildings_linked['_hhWt_n'])

**Note:** Could potentially check if this worked by creating a completely new column (instead of just replacing the old one), and seeing how the two compare.

### Step 14: Calculate remaining per-household capacity

**Equation 11 from Brockway et al.:**

`DER_remain`, or kW per household = `_hhWt_n` / `tothh_Cpoly`

In [ ]:
pge_buildings_linked['DER_remain'] = pge_buildings_linked['_hhWt_n'] / pge_buildings_linked['tothh_Cpoly'] * 1000

### Step 15: Calculate final per-household hosting capacity

**Equation 12 from Brockway et al.**

`DER` = `DER_exist` + `DER_remain`

In [ ]:
pge_buildings_linked_clean = pge_buildings_linked[['ID', 'unit', 'bbox', 'geometry', # columns used in analysis
                                                                        'circuit_id', 'segment_id', 'GenCapacit', 
                                                                        'dist_to_line_m','GEOID_right', 
                                                                        
                                                                        # columns created for analysis
                                                                        'length_m', 'units_by_tract', 'max_gen', 'total_feeder_length', 'perc_length',
                                                                        'home_count_seg', 'home_count_circuit', 'perc_homes',
                                                                        'tothh_Cpoly', 'DER_max_Cpoly', '_hhWt', '_hhWt_n', 'DER_remain']]

## Plot (like Brockway)

In [ ]:
pge_buildings_linked_clean.plot(column = 'DER_remain')

## GEOID check

In [ ]:
alameda_singlefamily_linked['GEOID_right'] == alameda_singlefamily_linked['GEOID_left']

In [ ]:
alameda_singlefamily_linked['GEOID_right'].eq(alameda_singlefamily_linked['GEOID_left']).value_counts()

In [ ]:
alameda_singlefamily_linked.head()

In [ ]:
alameda_singlefamily_linked['DER_remain'].describe()

In [ ]:
alameda_singlefamily_linked['DER_remain'].plot(kind = "hist",
                                              xlim=(0, 500),
                                              bins=range(0, 500, 25))

**Note:** Without Op Flex, no negatives! 

## Repeat workflow for generation capacity *with* operational flex

Starting with Step 4 --> the first place generation capacity is introduced.

In [ ]:
# replicate entire pipeline from above

# Step 4
# find maximum generation across feederline
max_gen = alameda_singlefamily_linked.groupby('circuit_id').max('GenCapac_1')

# save just the necessary column and rename
max_gen_feeder = max_gen['GenCapac_1'].rename('max_gen_opflex')

# join max generation by feeder to entire df
alameda_singlefamily_linked = alameda_singlefamily_linked.merge(
    max_gen_feeder,
    on = 'circuit_id')

# Step 10b
circ_poly_gen = alameda_singlefamily_linked.groupby(['GEOID_right', 'circuit_id']).max('GenCapac_1')

circ_poly_gen_max = circ_poly['GenCapac_1'].rename('DER_max_Cpoly_opflex')

alameda_singlefamily_linked = alameda_singlefamily_linked.merge(
    circ_poly_gen_max,
    on = ['GEOID_right', 'circuit_id'])

# Step 11
# calculate weight by household
alameda_singlefamily_linked['_hhWt_opflex'] = alameda_singlefamily_linked['DER_max_Cpoly_opflex'] * (alameda_singlefamily_linked['tothh_Cpoly'] / alameda_singlefamily_linked['home_count_circuit'])

# Step 12
# calculate summation of household-weighted max capacity across circuit
summ_hhWt = alameda_singlefamily_linked.groupby('circuit_id').sum('_hhWt_opflex')

# save the summed column and rename
summ_hhWt = summ_hhWt['_hhWt_opflex'].rename('summ_hhWt_opflex')

# add to data frame through join
alameda_singlefamily_linked = alameda_singlefamily_linked.merge(
    summ_hhWt,
    on = ['circuit_id'])

# equation 9
alameda_singlefamily_linked['_hhWt_n_opflex'] = alameda_singlefamily_linked['_hhWt_opflex'] * (alameda_singlefamily_linked['max_gen_opflex'] / alameda_singlefamily_linked['summ_hhWt_opflex'])

# Step 13
alameda_singlefamily_linked['_hhWt_n_opflex'] = np.where(
    
    # condition -- weighted generation is greater than max generation across circuit/feederline
    alameda_singlefamily_linked['_hhWt_n_opflex'] > alameda_singlefamily_linked['DER_max_Cpoly_opflex'],
    
    # replace with max generation of feederline if condition is true
    alameda_singlefamily_linked['DER_max_Cpoly_opflex'], 
    
    # otherwise, keep original calculated value
    alameda_singlefamily_linked['_hhWt_n_opflex'])

# Step 14
alameda_singlefamily_linked['DER_remain_opflex'] = alameda_singlefamily_linked['_hhWt_n_opflex'] / alameda_singlefamily_linked['tothh_Cpoly'] * 1000

#### Explore

In [ ]:
alameda_singlefamily_linked['DER_remain_opflex'].describe()

In [ ]:
alameda_singlefamily_linked['DER_remain'].describe()

**With** operational flex appears to have larger generation capacity values, and no negative ones (opposite of the raw exploration we did before...does this make sense?)

In [ ]:
alameda_singlefamily_linked['DER_remain_opflex'].plot(kind = "hist")

minor change